# phi4-langevin-ac on Google Colab (GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/droyktton/phi4-langevin-ac/blob/main/notebooks/colab_demo.ipynb)

This notebook shows how to **clone, compile and run** [phi4-langevin-ac](https://github.com/droyktton/phi4-langevin-ac)
on a Colab GPU: the Langevin dynamics of a circular domain in the disordered phi^4 model, driven by an AC field
at finite temperature. See the [README](https://github.com/droyktton/phi4-langevin-ac#readme) for the model and the output formats.

**Before you start:** `Runtime → Change runtime type → Hardware accelerator: GPU` (e.g. T4), then run the cells in order.
Colab GPU runtimes already ship the CUDA toolkit (`nvcc`), `make`, and the Python packages used by the analysis scripts.

## 1. Check the GPU and the CUDA compiler

In [ ]:
!nvidia-smi
!nvcc --version

## 2. Clone the repository

In [ ]:
import os
REPO = "https://github.com/droyktton/phi4-langevin-ac.git"
WORKDIR = "/content/phi4-langevin-ac" if os.path.isdir("/content") else os.path.abspath("phi4-langevin-ac")

if not os.path.isdir(WORKDIR):
    !git clone --depth 1 {REPO} {WORKDIR}
%cd {WORKDIR}
!git log -1 --oneline

## 3. Compile

The `Makefile` passes `ARCH` to `nvcc -arch`. We read the compute capability of the GPU that Colab assigned
(e.g. `7.5` for a T4 → `sm_75`) and compile natively for it.

In [ ]:
import subprocess
cc = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
).split()[0]
ARCH = "sm_" + cc.replace(".", "")
print("compiling for", ARCH)

!make clean -s && make ARCH={ARCH}
!./phi4langevin --help

## 4. Run a simulation

These are the parameters of the demo in the README (`scripts/run_demo.sh`): `L=512`, `R=150`, strong disorder `Delta=1`,
`T=0.02`, AC field `h0=0.3`, `f=0.004` (period 250), for three periods. Walls are written every `tout=5` and full-field
snapshots every `tconf=10` (~2.4 MB each, ~190 MB in total). Edit them freely; `--help` above lists all the options.

In [ ]:
params = dict(L=512, radius=150, delta=1.0, T=0.02, h0=0.3, freq=0.004,
              dt=0.05, tmax=750, tout=5, tconf=10, seed=1)
params["noise-seed"] = 2

OUT = os.path.join(WORKDIR, "demo")
os.makedirs(OUT, exist_ok=True)
%cd {OUT}
!rm -f walls_*.dat timeseries_*.dat config_t*.dat contours_* *.png

args = " ".join(f"--{k} {v}" for k, v in params.items())
RUN = f"seed{params['seed']}_nseed{params['noise-seed']}"
print("./phi4langevin", args)
!time {WORKDIR}/phi4langevin {args}
!ls {OUT} | head; echo ...; du -sh {OUT}

## 5. Post-analysis

- `plot_wall_evolution.py`: wall crossings, `R_eff(t)` with `h(t)`, and the polar profile `r(theta)` (no snapshots needed).
- `extract_contours.py`: the `phi=0` level curves of every snapshot, as ordered closed polylines stitched across the periodic boundaries.
- `plot_contours.py`: snapshot grid, number of curves / wall length / `+` area vs `t`, and a GIF.

In [ ]:
!python3 {WORKDIR}/scripts/plot_wall_evolution.py walls_{RUN}.dat --ntimes 8
!python3 {WORKDIR}/scripts/extract_contours.py . --min-length 8 --run {RUN}
!python3 {WORKDIR}/scripts/plot_contours.py contours_{RUN}.dat --panels 16 --gif

## 6. Results

In [ ]:
from IPython.display import Image, display
for f in [f"{RUN}_evolution.png", f"contours_{RUN}_grid.png", f"contours_{RUN}_ncurves.png"]:
    print(f)
    display(Image(filename=f))

In [ ]:
display(Image(filename=f"contours_{RUN}.gif"))

### Your own analysis

`timeseries_<run>.dat` has one line per output time, with columns `t h(t) <phi> A+ R_eff n_wall xcm ycm <r> var(r)`,
and `contours_<run>.csv` has one row per `(t, curve)`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ts = np.loadtxt(f"timeseries_{RUN}.dat")
t, h, Reff, var_r = ts[:, 0], ts[:, 1], ts[:, 4], ts[:, 9]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(t, Reff, label=r"$R_{\rm eff}$")
ax2 = ax[0].twinx(); ax2.plot(t, h, color="C1", alpha=0.6); ax2.set_ylabel("h(t)", color="C1")
ax[0].set_xlabel("t"); ax[0].set_ylabel(r"$R_{\rm eff}$")
ax[1].plot(t, var_r); ax[1].set_xlabel("t"); ax[1].set_ylabel("var(r)  (wall roughness)")
plt.tight_layout(); plt.show()

## 7. Download the results (optional)

The Colab filesystem is deleted when the runtime is recycled. This zips the light outputs (walls, time series,
contours, figures) and downloads them; add `config_t*.dat` if you also want the full-field snapshots.

In [ ]:
!zip -q results_{RUN}.zip logfile.dat walls_{RUN}.dat timeseries_{RUN}.dat contours_{RUN}.* *.png
try:
    from google.colab import files
    files.download(f"results_{RUN}.zip")
except ImportError:
    print("not running on Colab; the zip is in", OUT)